# Comparaison BMOO cohérente sur grille fixe

Comparaison EHVI vs PAL sur GPmp et BoTorch : GPmp compare `MC_sMC` (estimation Monte-Carlo, recodée intégralement, validée numériquement contre la version analytique) et `EHVI_explicit` (analytique, 2 ou 3 objectifs) ; les deux backends ajoutent `PAL` (Pareto Active Learning, classification par hyperrectangles de confiance). Une seule grille maximin-LHS (gpmp) est construite par problème. Les répétitions changent uniquement les points initiaux. Estimation des hyperparametres du GP par maximum de vraisemblance pur (ML), identique pour les deux backends : beta (moyenne), sigma^2 (variance) et rho (portee) sont estimes ainsi ; nu (regularite Matern) est fixe, pas estime.

Problèmes : suites ZDT/DTLZ/WFG (via pymoo), pas des objectifs synthétisés à partir d'une seule fonction scalaire (voir README.md pour pourquoi cette ancienne construction a été abandonnée).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
sys.path.insert(0, str(Path.cwd().parent / "src"))


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import compare, grid_ablation, candidate_ablation
from problem import ZDT, DTLZ, WFG
Path("results").mkdir(exist_ok=True)

## Paramètres


In [ ]:
N_REPEATS=10
N_ITER=25
N_INIT=6
N_CANDIDATES=4096
SEED=0
# One representative problem per family, m in the range it supports (ZDT: 2
# objectives only). The full families (5 ZDT + 7 DTLZ + 9 WFG) are available
# as problem.ZDT/DTLZ/WFG for a full sweep via compare.py's __main__.
BASE_PROBLEMS=[(ZDT[0],(2,)),(ZDT[1],(2,)),(DTLZ[1],(2,3)),(DTLZ[6],(2,3)),(WFG[3],(2,3)),(WFG[8],(2,3))]
BASE_PROBLEMS_BY_NAME={base.name:base for base,_ in BASE_PROBLEMS}

## Benchmark sur ZDT/DTLZ/WFG

In [ ]:
runs={}
for base,ms in BASE_PROBLEMS:
    for m in ms:
        problem=base.with_n_obj(m)
        results,times,metadata=compare.run(problem,N_REPEATS,N_ITER,N_INIT,N_CANDIDATES,SEED)
        compare.save(problem,results,times,metadata)
        compare.summarize(problem,results,times)
        runs[(problem.name,m)]=(results,times,metadata)

## Vérification du protocole


In [ ]:
for (name,m),(_,_,meta) in runs.items():
    starts=meta["initial_indices"]
    print(name,m,"grid",meta["grid"].shape,"initial sets",starts.shape,"distinct",len(np.unique(starts,axis=0)))


## Courbes et temps de calcul


In [ ]:
for (name,m),(results,times,_) in runs.items():
    problem=BASE_PROBLEMS_BY_NAME[name].with_n_obj(m)
    compare.plot(problem,results,N_INIT,N_ITER)
    plt.show()
    compare.summarize(problem,results,times)

## Front de Pareto atteint vs vrai front

Front atteint par chaque algorithme (une répétition) superposé au vrai front continu du problème (pymoo). Repere visuel uniquement -- les metriques restent calculees contre l'oracle restreint a la grille.

In [ ]:
for (name,m),(_,_,metadata) in runs.items():
    problem=BASE_PROBLEMS_BY_NAME[name].with_n_obj(m)
    try:
        compare.plot_pareto_fronts(problem,metadata)
        plt.show()
    except NotImplementedError:
        print(name,m,"has no known Pareto front in pymoo")

## Ablation GPmp de la taille de grille


In [ ]:
problem=DTLZ[1].with_n_obj(2)
histories,ablation_times,meta=grid_ablation.run(problem,grid_sizes=(256,512,1024,2048,4096),n_repeats=N_REPEATS,n_init=N_INIT,n_iter=N_ITER,seed=SEED)
grid_ablation.plot(problem,histories,n_init=N_INIT,path="results/dtlz2_grid_ablation.png")
plt.show()

## Ablation de la strategie de candidats (EHVI) : LHS vs Sobol vs Gradient (BFGS)

Meme protocole EHVI (`explicit_ehvi` cote GPmp, `EHVI` cote BoTorch, pas d'Optuna -- TPE n'a pas d'EHVI analytique), 3 strategies de selection du prochain point : grille LHS fixe, Sobol redessine a chaque iteration, optimisation continue L-BFGS-B (`optimizer="gradient"` des deux cotes). Voir `candidate_ablation.py`.

In [ ]:
problem=ZDT[0].with_n_obj(2)
candidate_results,candidate_meta=candidate_ablation.run_all(
    problem,n_repeats=N_REPEATS,n_init=N_INIT,n_iter=N_ITER,seed=SEED
)
candidate_ablation.plot(problem,candidate_results,n_init=N_INIT,
                        path="results/zdt1_candidate_ablation.png")
plt.show()
candidate_ablation.plot_pareto_fronts(problem,candidate_meta,
                                       path="results/zdt1_candidate_ablation_pareto.png")
plt.show()

## Comparaison GPmp : MC-sMC, EHVI explicite, PAL

La méthode explicite apparaît automatiquement uniquement pour 2 et 3 objectifs. Filtre sur la librairie "GPmp" uniquement, donc les trois critères GPmp apparaissent ensemble.

In [ ]:
problem=ZDT[0].with_n_obj(2)
results,times,metadata=compare.load("results/zdt1_2obj_histories.npz")
compare.plot(problem,results,n_init=N_INIT)
compare.summarize(problem,results,times)
plt.show()

## Comparaison GPmp MC-sMC et EHVI explicite

La méthode explicite apparaît automatiquement uniquement pour 2 et 3 objectifs.


In [ ]:
for m in (2, 3):
    problem = DTLZ[1].with_n_obj(m)
    results, times, metadata = compare.run(problem, N_REPEATS, N_ITER, N_INIT, N_CANDIDATES, SEED)
    gpmp_results = {k: v for k, v in results.items() if k[0] == "GPmp"}
    compare.plot(problem, gpmp_results, N_INIT, N_ITER)
    plt.show()